In [ ]:
import sys
sys.path.append('..')

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import pandas as pd
import numpy as np

from src.config import PATHS
from src.viz import set_style, save_fig, annotate_source

set_style()

In [ ]:
df_h = pd.read_parquet(PATHS['analytical'] / 'df_hogar_analytic.parquet')
df_p = pd.read_parquet(PATHS['analytical'] / 'df_persona_analytic.parquet')

print(f'Hogares analíticos : {df_h.shape[0]:,} filas × {df_h.shape[1]} columnas')
print(f'Personas analíticas: {df_p.shape[0]:,} filas × {df_p.shape[1]} columnas')

## 7.2 Variables Cuantitativas: Medidas de Tendencia Central y Dispersión

In [ ]:
num_vars = ['n_personas_hogar', 'n_usuarios_internet', 'prop_usuarios_internet', 'prob_pobreza_multidimensional', 'fex_c']

resumen_num = []
for var in num_vars:
 s = df_h[var].dropna()
 resumen_num.append({
 'Variable': var,
 'N': len(s),
 'Media (x̄)': s.mean(),
 'Mediana (Me)': s.median(),
 'Desv. Est. (SD)': s.std(),
 'Q1': s.quantile(0.25),
 'Q3': s.quantile(0.75),
 'IQR': s.quantile(0.75) - s.quantile(0.25),
 'Mínimo': s.min(),
 'Máximo': s.max()
 })

df_resumen_num = pd.DataFrame(resumen_num).round(3)
df_resumen_num

## 7.3 Distribución de la Proporción de Usuarios de Internet por Hogar

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.8))
sns.histplot(df_h['prop_usuarios_internet'] * 100, bins=20, kde=True, color='#2b5c8f', ax=ax, edgecolor='white')
ax.set_title('Distribución de la Proporción de Usuarios de Internet por Hogar', pad=15)
ax.set_xlabel('Proporción de miembros que usan Internet (%)', labelpad=8)
ax.set_ylabel('Frecuencia (Nº de Hogares)')
plt.tight_layout()
annotate_source(ax, 'Fuente: DANE — ECV 2024. Muestra: 85,387 hogares.')
save_fig(fig, 'fig05_prop_usuarios_distribucion')
plt.show()

## 7.4 Variables Categóricas: Nivel Hogar (Frecuencias Absolutas y Relativas)

In [ ]:
def tabla_frecuencias(df, col, nombre_var=''):
 vc = df[col].value_counts(dropna=False)
 pct = (df[col].value_counts(dropna=False, normalize=True) * 100).round(2)
 tab = pd.DataFrame({'Frecuencia (n_i)': vc, 'Porcentaje (%)': pct})
 tab.index.name = nombre_var or col
 return tab

print(tabla_frecuencias(df_h, 'acceso_internet_hogar'))

print(tabla_frecuencias(df_h, 'tiene_computador'))

print(tabla_frecuencias(df_h, 'zona'))

print(tabla_frecuencias(df_h, 'sit_eco_label'))

print(tabla_frecuencias(df_h, 'jefe_nivel_educ'))

### Gráfico 1: Acceso a Internet vs. Tenencia de Computador en Hogares

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.8))
indicadores = pd.Series({
 'Hogares con acceso a Internet': df_h['acceso_internet_hogar'].mean() * 100,
 'Hogares con computador (mesa/portátil/tablet)': df_h['tiene_computador'].mean() * 100,
 'Hogares con conexión fija': df_h['tiene_internet_fijo'].mean() * 100,
 'Hogares con conexión móvil': df_h['tiene_internet_movil'].mean() * 100
})
colores = ['#1f77b4', '#aec7e8', '#2ca02c', '#98df8a']
bars = ax.barh(indicadores.index, indicadores.values, color=colores, height=0.55)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_xlim(0, 105)
ax.set_title('Acceso a Internet y Equipamiento Tecnológico en los Hogares', pad=15)
ax.set_xlabel('Porcentaje de hogares (%)', labelpad=8)
for b in bars:
 w = b.get_width()
 ax.text(w + 1.2, b.get_y() + b.get_height()/2, f'{w:.1f}%', va='center', fontweight='bold', fontsize=9.5)
plt.tight_layout()
annotate_source(ax, 'Fuente: DANE — Encuesta de Calidad de Vida (ECV) 2024. Muestra: 85,387 hogares.')
save_fig(fig, 'fig01_acceso_computador_hogares')
plt.show()

### Gráfico 2: Distribución de Hogares según Percepción de Dificultad Económica

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.8))
sit_eco = (df_h['sit_eco_label'].value_counts(normalize=True).sort_index() * 100)
paleta_eco = ['#2ca02c', '#8c564b', '#ff7f0e', '#d62728', '#7f0f15']
bars = ax.bar(sit_eco.index, sit_eco.values, color=paleta_eco, width=0.55, edgecolor='black', alpha=0.85)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title('Distribución de Hogares según Percepción de Dificultad Económica', pad=15)
ax.set_ylabel('Porcentaje de hogares (%)')
ax.set_xlabel('Facilidad / Dificultad para cubrir gastos mínimos', labelpad=10)
for b in bars:
 h = b.get_height()
 ax.text(b.get_x() + b.get_width()/2, h + 0.8, f'{h:.1f}%', ha='center', fontweight='bold', fontsize=9)
plt.tight_layout()
annotate_source(ax, 'Fuente: DANE — ECV 2024. Pregunta P3353. Muestra: 85,387 hogares.')
save_fig(fig, 'fig02_distribucion_dificultad_economica')
plt.show()

## 7.5 Variables Categóricas: Nivel Persona (Uso Individual y Modalidad)

In [ ]:
print(tabla_frecuencias(df_p, 'usa_internet'))

print(tabla_frecuencias(df_p, 'tipo_conexion'))

print(tabla_frecuencias(df_p, 'frecuencia_uso'))

print(tabla_frecuencias(df_p, 'nivel_educ_agrupado'))

### Gráfico 3: Modalidad de Conexión a Internet en las Personas

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.8))
con_tipo = (df_p['tipo_conexion'].value_counts(normalize=True) * 100)
bars = ax.barh(con_tipo.index, con_tipo.values, color='#3182bd', height=0.55)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_xlim(0, 75)
ax.set_title('Modalidad de Conexión a Internet en las Personas (Últimos 12 meses)', pad=15)
ax.set_xlabel('Porcentaje de personas (%)', labelpad=8)
for b in bars:
 w = b.get_width()
 ax.text(w + 1.0, b.get_y() + b.get_height()/2, f'{w:.1f}%', va='center', fontweight='bold', fontsize=9.5)
plt.tight_layout()
annotate_source(ax, 'Fuente: DANE — ECV 2024. Módulo TIC. Muestra: 214,607 personas.')
save_fig(fig, 'fig03_tipo_conexion_personas')
plt.show()

### Gráfico 4: Frecuencia de Uso de Internet en las Personas

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.8))
frec_s = (df_p['frecuencia_uso'].value_counts(normalize=True).sort_index() * 100)
bars = ax.barh(frec_s.index, frec_s.values, color='#08519c', height=0.55)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_xlim(0, 75)
ax.set_title('Frecuencia de Uso de Internet en las Personas', pad=15)
ax.set_xlabel('Porcentaje de personas (%)', labelpad=8)
for b in bars:
 w = b.get_width()
 ax.text(w + 1.0, b.get_y() + b.get_height()/2, f'{w:.1f}%', va='center', fontweight='bold', fontsize=9.5)
plt.tight_layout()
annotate_source(ax, 'Fuente: DANE — ECV 2024. Módulo TIC (P769). Muestra: 214,607 personas.')
save_fig(fig, 'fig04_frecuencia_uso_internet')
plt.show()